[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mbanuelos/grad_math_modeling/blob/main/Lectures/Module3_DeepLearning/11_DecisionTrees.ipynb#copy=true)


# Decision Trees: Nonlinear, Interpretable Models

## Learning objectives

By the end of this lesson, you should be able to:

- explain how a decision tree divides feature space with sequential questions;
- describe Gini impurity and how it guides a split;
- fit, visualize, and evaluate a classification tree;
- explain how tree depth affects overfitting.

**Student Learning Outcome:** Fit a nonlinear supervised-learning model and justify evaluation choices using held-out data.


## Why a nonlinear model?

Linear models use one weighted combination of features. A decision tree asks sequential threshold questions such as ``is alcohol at most 13.2?'' Every root-to-leaf path defines a region of feature space, so trees represent interactions and nonlinear decision boundaries without manually adding polynomial features.

We will classify three wine cultivars from chemical measurements. The class label is known for every observation, so this is supervised classification.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split

RANDOM_STATE = 232
wine = load_wine(as_frame=True)
X, y = wine.data, wine.target
class_names = wine.target_names
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)
print("features:", list(X.columns))
print("training observations:", len(X_train), "test observations:", len(X_test))

X.head()


## Part 1 — Nodes, leaves, and impurity

A **node** contains a subset of training observations. A **split** sends observations left or right using one feature threshold. A terminal node is a **leaf**; its predicted class is the majority class among training observations in that leaf.

For class proportions $p_1,\ldots,p_C$, Gini impurity is

$$G=1-\sum_{c=1}^C p_c^2.$$

It is zero when every observation in the node has the same class. The tree selects splits that lower weighted child-node impurity.


### Checkpoint 1

A node contains 8 observations: 6 from class 0 and 2 from class 1. Is its Gini impurity zero? Would two pure child nodes lower impurity? Explain.

> **YOUR ANSWER:**


## Part 2 — Fit and inspect a small tree

We limit `max_depth` so that the tree remains readable. This is a modeling choice, not a property of the data. The test set remains untouched while the tree is fitted.


In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

tree = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)
tree.fit(X_train, y_train)
test_predictions = tree.predict(X_test)
print("test accuracy:", round(accuracy_score(y_test, test_predictions), 3))
print("number of leaves:", tree.get_n_leaves())

plt.figure(figsize=(18, 8))
plot_tree(tree, feature_names=X.columns, class_names=class_names,
          filled=True, rounded=True, proportion=True)
plt.show()


### Practice 2

Choose one root-to-leaf path in the plot. Translate its threshold questions into an ordinary-language rule. What class does its leaf predict, and what evidence in the plot supports your answer?

> **YOUR ANSWER:**


In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, test_predictions, display_labels=class_names)
plt.title("Decision-tree test predictions")
plt.show()


## Part 3 — Flexibility versus generalization

A deeper tree can carve training data into more regions. This can reduce training error while increasing test error. The table below is exploratory; use cross-validation rather than repeatedly selecting the best final test result in a real workflow.


In [ ]:
rows = []
for depth in range(1, 9):
    candidate = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE).fit(X_train, y_train)
    rows.append({"max depth": depth, "training accuracy": candidate.score(X_train, y_train),
                 "test accuracy": candidate.score(X_test, y_test), "leaves": candidate.get_n_leaves()})
pd.DataFrame(rows).round(3)


### Practice 3

Identify a depth where training accuracy is noticeably higher than test accuracy. What pattern would count as evidence of overfitting, and why would cross-validation be preferable for selecting a final depth?

> **YOUR ANSWER:**


## Lesson summary

Decision trees create nonlinear decision regions through interpretable threshold rules. Their flexibility is useful, but unrestricted trees can memorize training noise. Next, random forests reduce the instability of one tree by aggregating many randomized trees.
